# 06 — Neural Net + Forecast (Phase 6)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 6: import และเรียกใช้ฟังก์ชันจาก `src/rri/features/`
และ `src/rri/models/` เท่านั้น — ไม่มี business logic ในเซลล์ ตาม CLAUDE.md N2

**ต่อจาก Phase 5 checkpoint** (business-sense review ผ่านแล้ว 2026-08-24, ดู
`docs/phase5_modeling_findings.md`) — baseline (GLM + LightGBM) มั่นคงพอที่จะสร้าง neural
net ต่อ

ลำดับขั้นตอน:

1. MLP + entity embeddings (severity classifier) เทียบกับ LightGBM classifier
2. Province embeddings → t-SNE 2 มิติ
3. Global GRU forecast (6 เดือนล่วงหน้า) เทียบกับ seasonal-naive baseline
4. ตารางเปรียบเทียบรวมทุก tier + สรุปข้อสรุป
5. บันทึกโมเดลทั้งหมดสำหรับ Phase 7 (dashboard)

## 0. Setup

In [1]:
import os

# PyTorch and LightGBM both bundle their own libomp runtime -- this kernel process
# uses both (rri.models.mlp_entity_embeddings/gru_forecast AND
# rri.models.lightgbm_baseline), which without this reliably segfaults inside
# LightGBM's Dataset construction (see tests/conftest.py for the full story --
# there it's solved by running the two test groups in separate processes; a
# single notebook kernel can't split into two processes, so the env-var
# mitigation is this notebook's only option, set here before either library
# is imported).
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import duckdb
import numpy as np
import polars as pl
from sklearn.metrics import confusion_matrix, f1_score

from config.settings import settings
from rri.features.frequency_severity import build_frequency_severity_frame
from rri.features.severity_classification import build_severity_classification_frame
from rri.features.splits import assert_time_based_split, time_based_split
from rri.models import persistence as per
from rri.models.glm_baseline import (
    compute_flood_fill_value,
    fit_gamma_severity_glm,
    fit_poisson_frequency_glm,
    frequency_training_rows,
    predict_glm,
    severity_training_rows,
)
from rri.models.gru_forecast import (
    build_gru_windows,
    build_province_month_series,
    fit_gru_forecast,
    predict_gru_forecast,
    seasonal_naive_forecast,
)
from rri.models.gru_forecast import (
    split_windows as split_gru_windows,
)
from rri.models.lightgbm_baseline import (
    fit_lightgbm_frequency,
    fit_lightgbm_severity,
    fit_lightgbm_severity_classifier,
    predict_lightgbm_frequency,
    predict_lightgbm_severity,
    predict_lightgbm_severity_classifier,
)
from rri.models.metrics import (
    gamma_deviance,
    gini_coefficient,
    mean_absolute_error,
    poisson_deviance,
)
from rri.models.mlp_entity_embeddings import (
    fit_mlp_classifier,
    predict_mlp_classifier,
    project_province_embeddings_2d,
    province_embedding_table,
    save_province_embedding_plot,
)
from rri.utils.logging import configure_logging
from rri.warehouse.dim_vehicle import vehicle_group_mapping_path

configure_logging()
reports_dir = settings.reports_dir
reports_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(settings.warehouse_db_path), read_only=True)
fact_accident = con.sql("SELECT * FROM fact_accident").pl()
dim_date = con.sql("SELECT * FROM dim_date").pl()
dim_geography = con.sql("SELECT * FROM dim_geography").pl()
dim_vehicle = con.sql("SELECT * FROM dim_vehicle").pl()
dim_weather_band = con.sql("SELECT * FROM dim_weather_band").pl()
con.close()

rainfall_df = pl.read_parquet(settings.staging_dir / "rainfall.parquet")
stt_car_fuel_df = pl.read_parquet(settings.staging_dir / "stt_car_fuel.parquet")
car_registered_new_df = pl.read_parquet(settings.staging_dir / "car_registered_new.parquet")
accidents_staging = pl.read_parquet(settings.staging_dir / "accidents.parquet")
vehicle_mapping_path = vehicle_group_mapping_path()

freq_sev_frame = build_frequency_severity_frame(
    fact_accident,
    dim_date,
    dim_geography,
    dim_vehicle,
    rainfall_df,
    stt_car_fuel_df,
    car_registered_new_df,
    vehicle_mapping_path,
)
sev_class_frame = build_severity_classification_frame(
    accidents_staging, fact_accident, dim_date, dim_geography, dim_vehicle, dim_weather_band
)
print("frequency_severity frame:", freq_sev_frame.shape)
print("severity_classification frame:", sev_class_frame.shape)

frequency_severity frame: (18480, 38)
severity_classification frame: (131672, 31)


## 1. MLP + Entity Embeddings vs. LightGBM (severity classifier)

Frame เดียวกัน (record-level), target เดียวกัน (severity 4 ระดับ), time split เดียวกัน —
ต่างกันแค่โมเดล (CLAUDE.md Step 5: "fair comparison, not to force a win")

In [2]:
sev_class_split = time_based_split(sev_class_frame)
for name, split in [("severity_classification", sev_class_split)]:
    assert_time_based_split(split)
    print(f"{name}: train={split.train.height} valid={split.valid.height} test={split.test.height}")

severity_classification: train=84005 valid=23952 test=23715


In [3]:
t0 = time.time()
lgbm_clf = fit_lightgbm_severity_classifier(sev_class_split.train, sev_class_split.valid)
lgbm_clf_time = time.time() - t0
lgbm_clf_pred = predict_lightgbm_severity_classifier(lgbm_clf, sev_class_split.test)
lgbm_clf_y = sev_class_split.test["severity_class"].to_numpy()
lgbm_clf_f1 = f1_score(lgbm_clf_y, lgbm_clf_pred, average="macro")
print(f"LightGBM classifier -- macro-F1 (test): {lgbm_clf_f1:.3f} (fit time {lgbm_clf_time:.1f}s)")
print(confusion_matrix(lgbm_clf_y, lgbm_clf_pred, labels=sorted(set(lgbm_clf_y))))

/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM classifier -- macro-F1 (test): 0.398 (fit time 3.7s)
[[ 803  986  552   46]
 [ 471 3723 4616   99]
 [ 118 2123 8527   26]
 [ 232  818  520   55]]


In [4]:
t0 = time.time()
mlp_clf = fit_mlp_classifier(sev_class_split.train, sev_class_split.valid)
mlp_clf_time = time.time() - t0
mlp_clf_pred = predict_mlp_classifier(mlp_clf, sev_class_split.test)
mlp_clf_f1 = f1_score(lgbm_clf_y, mlp_clf_pred, average="macro")
print(
    f"MLP classifier -- macro-F1 (test): {mlp_clf_f1:.3f} "
    f"(fit time {mlp_clf_time:.1f}s, best_epoch={mlp_clf.best_epoch}/{len(mlp_clf.train_losses)})"
)
print(confusion_matrix(lgbm_clf_y, mlp_clf_pred, labels=sorted(set(lgbm_clf_y))))

MLP classifier -- macro-F1 (test): 0.382 (fit time 4.3s, best_epoch=25/41)
[[ 728 1142  517    0]
 [ 412 3774 4723    0]
 [  85 1892 8817    0]
 [ 228  884  513    0]]


## 2. Province Risk-Profile Embeddings → t-SNE

การ์ดใบนี้เป็น deliverable อิสระ ไม่ขึ้นกับว่า MLP จะชนะ LightGBM หรือไม่ (CLAUDE.md Step 5)

In [5]:
province_embeddings = province_embedding_table(mlp_clf)
projected = project_province_embeddings_2d(province_embeddings)
plot_path = save_province_embedding_plot(
    projected, dim_geography, reports_dir / "province_embeddings_tsne.png"
)
print("saved", plot_path)
projected.head(5)

saved /Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/reports/province_embeddings_tsne.png


province_th,tsne_x,tsne_y
str,f32,f32
"""กระบี่""",6.24236,3.183836
"""กรุงเทพมหานคร""",-7.232054,5.520893
"""กาญจนบุรี""",-1.161013,-6.426793
"""กาฬสินธุ์""",-7.643723,4.2611
"""กำแพงเพชร""",5.124429,5.719285


## 3. Global GRU Forecast vs. Seasonal-Naive

Grain: จังหวัด × เดือน (รวมทุกกลุ่มรถ) — 12 เดือนย้อนหลัง พยากรณ์ 6 เดือนข้างหน้า (48 เดือน
x 77 จังหวัด ข้อมูลค่อนข้างบางสำหรับ sequence model, CLAUDE.md Step 6 บอกให้ scope เบาๆ
ไม่ใช่จุดเด่นหลัก)

In [6]:
province_month_series = build_province_month_series(freq_sev_frame)
gru_windows = build_gru_windows(province_month_series)
gru_splits = split_gru_windows(gru_windows)
for name, w in gru_splits.items():
    target_years = sorted(set(w.target_start_year.tolist()))
    print(f"{name}: {len(w.province)} windows (target year {target_years})")

train: 539 windows (target year [2023])
valid: 539 windows (target year [2024])
test: 539 windows (target year [2025])


In [7]:
t0 = time.time()
gru_fitted = fit_gru_forecast(gru_splits["train"], gru_splits["valid"])
gru_time = time.time() - t0
gru_pred_test = predict_gru_forecast(gru_fitted, gru_splits["test"])
y_test_gru = gru_splits["test"].y

gru_deviance = poisson_deviance(y_test_gru.flatten(), gru_pred_test.flatten())
gru_mae = mean_absolute_error(y_test_gru.flatten(), gru_pred_test.flatten())
print(
    f"GRU -- Poisson deviance (test): {gru_deviance:.3f}, MAE: {gru_mae:.3f} "
    f"(fit time {gru_time:.1f}s, best_epoch={gru_fitted.best_epoch}/{len(gru_fitted.train_losses)})"
)

GRU -- Poisson deviance (test): 14.640, MAE: 13.589 (fit time 1.9s, best_epoch=513/614)


In [8]:
naive_pred_test = seasonal_naive_forecast(province_month_series, gru_splits["test"])
naive_mask = ~np.isnan(naive_pred_test)
naive_deviance = poisson_deviance(y_test_gru[naive_mask], naive_pred_test[naive_mask])
naive_mae = mean_absolute_error(y_test_gru[naive_mask], naive_pred_test[naive_mask])
print(f"Seasonal-naive -- Poisson deviance (test): {naive_deviance:.3f}, MAE: {naive_mae:.3f}")
print(f"Seasonal-naive coverage: {naive_mask.mean():.0%}")

gru_beats_naive = gru_deviance < naive_deviance
print(f"\nGRU beats seasonal-naive: {gru_beats_naive}")

Seasonal-naive -- Poisson deviance (test): 7.396, MAE: 10.150
Seasonal-naive coverage: 100%

GRU beats seasonal-naive: False


## 4. GLM + LightGBM frequency/severity refit (for the consolidated table)

Refit บนหน้าต่างเวลาเดียวกัน เพื่อให้ตาราง comparison มีทุก tier ในโน้ตบุ๊กเดียว (Phase 5
notebook มีตัวเลขเหล่านี้แล้ว แต่คนละ kernel/session — reproduce ที่นี่เพื่อความสมบูรณ์ของ
ตารางสุดท้าย)

In [9]:
freq_sev_split = time_based_split(freq_sev_frame)
freq_train = frequency_training_rows(freq_sev_split.train)
freq_valid = frequency_training_rows(freq_sev_split.valid)
freq_test = frequency_training_rows(freq_sev_split.test)
flood_fill = compute_flood_fill_value(freq_train)

t0 = time.time()
glm_freq = fit_poisson_frequency_glm(freq_train, flood_fill)
glm_freq_time = time.time() - t0
glm_freq_pred = predict_glm(glm_freq, freq_test, flood_fill)
glm_freq_deviance = poisson_deviance(freq_test["accident_count"].to_numpy(), glm_freq_pred)
glm_freq_gini = gini_coefficient(freq_test["accident_count"].to_numpy(), glm_freq_pred)

t0 = time.time()
lgbm_freq = fit_lightgbm_frequency(freq_train, freq_valid)
lgbm_freq_time = time.time() - t0
lgbm_freq_pred = predict_lightgbm_frequency(lgbm_freq, freq_test)
lgbm_freq_deviance = poisson_deviance(freq_test["accident_count"].to_numpy(), lgbm_freq_pred)
lgbm_freq_gini = gini_coefficient(freq_test["accident_count"].to_numpy(), lgbm_freq_pred)

sev_train = severity_training_rows(freq_sev_split.train)
sev_valid = severity_training_rows(freq_sev_split.valid)
sev_test = severity_training_rows(freq_sev_split.test)

t0 = time.time()
glm_sev = fit_gamma_severity_glm(sev_train, flood_fill)
glm_sev_time = time.time() - t0
glm_sev_pred = predict_glm(glm_sev, sev_test, flood_fill)
glm_sev_deviance = gamma_deviance(sev_test["severity_index_coarse"].to_numpy(), glm_sev_pred)
glm_sev_gini = gini_coefficient(sev_test["severity_index_coarse"].to_numpy(), glm_sev_pred)

t0 = time.time()
lgbm_sev = fit_lightgbm_severity(sev_train, sev_valid)
lgbm_sev_time = time.time() - t0
lgbm_sev_pred = predict_lightgbm_severity(lgbm_sev, sev_test)
lgbm_sev_deviance = gamma_deviance(sev_test["severity_index_coarse"].to_numpy(), lgbm_sev_pred)
lgbm_sev_gini = gini_coefficient(sev_test["severity_index_coarse"].to_numpy(), lgbm_sev_pred)

print("frequency/severity GLM + LightGBM refit done")

frequency/severity GLM + LightGBM refit done


/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


## 5. Consolidated Comparison Table → `reports/`

ทุก tier บน test set เดียวกัน (2025), พร้อม training time และระดับ explainability (เชิงคุณภาพ)

In [10]:
comparison = pl.DataFrame(
    {
        "tier": [
            "L0 GLM",
            "L1 LightGBM",
            "L0 GLM",
            "L1 LightGBM",
            "L1 LightGBM",
            "L2 MLP+embeddings",
            "L3 GRU",
            "L3 seasonal-naive",
        ],
        "task": [
            "frequency",
            "frequency",
            "severity",
            "severity",
            "severity classifier",
            "severity classifier",
            "6-month forecast",
            "6-month forecast",
        ],
        "metric_name": [
            "poisson_deviance",
            "poisson_deviance",
            "gamma_deviance",
            "gamma_deviance",
            "macro_f1",
            "macro_f1",
            "poisson_deviance",
            "poisson_deviance",
        ],
        "metric_value": [
            glm_freq_deviance,
            lgbm_freq_deviance,
            glm_sev_deviance,
            lgbm_sev_deviance,
            lgbm_clf_f1,
            mlp_clf_f1,
            gru_deviance,
            naive_deviance,
        ],
        "gini": [
            glm_freq_gini,
            lgbm_freq_gini,
            glm_sev_gini,
            lgbm_sev_gini,
            None,
            None,
            None,
            None,
        ],
        "train_time_seconds": [
            glm_freq_time,
            lgbm_freq_time,
            glm_sev_time,
            lgbm_sev_time,
            lgbm_clf_time,
            mlp_clf_time,
            gru_time,
            0.0,
        ],
        "explainability": [
            "high (coefficients)",
            "high (SHAP)",
            "high (coefficients)",
            "high (SHAP)",
            "high (SHAP)",
            "medium (embeddings + t-SNE, no per-prediction SHAP this phase)",
            "low (black-box sequence weights)",
            "trivial (lag-12 lookup)",
        ],
    }
)
comparison.write_csv(reports_dir / "phase6_full_model_comparison.csv")
comparison

tier,task,metric_name,metric_value,gini,train_time_seconds,explainability
str,str,str,f64,f64,f64,str
"""L0 GLM""","""frequency""","""poisson_deviance""",16.665911,0.751378,0.040807,"""high (coefficients)"""
"""L1 LightGBM""","""frequency""","""poisson_deviance""",27.54029,0.772877,0.145223,"""high (SHAP)"""
"""L0 GLM""","""severity""","""gamma_deviance""",1.213328,0.498792,0.056147,"""high (coefficients)"""
"""L1 LightGBM""","""severity""","""gamma_deviance""",1.208744,0.511317,0.039478,"""high (SHAP)"""
"""L1 LightGBM""","""severity classifier""","""macro_f1""",0.397851,null,3.68537,"""high (SHAP)"""
"""L2 MLP+embeddings""","""severity classifier""","""macro_f1""",0.382269,null,4.287334,"""medium (embeddings + t-SNE, no…"
"""L3 GRU""","""6-month forecast""","""poisson_deviance""",14.640052,null,1.946307,"""low (black-box sequence weight…"
"""L3 seasonal-naive""","""6-month forecast""","""poisson_deviance""",7.396415,null,0.0,"""trivial (lag-12 lookup)"""


## 6. สรุปข้อสรุป

- **Frequency**: GLM ชนะทั้ง deviance และ Gini ในโน้ตบุ๊ก Phase 5 ดั้งเดิม — ผลลัพธ์ที่นี่
  (refit ใน kernel เดียวกับ Phase 6) ควรสอดคล้องกัน ข้อมูล train มีแค่ 2 ปี (2022-2023)
  ซึ่งอาจไม่พอให้ LightGBM ใช้ความยืดหยุ่นได้เต็มที่
- **Severity**: LightGBM ชนะเล็กน้อยทั้ง deviance และ Gini
- **Severity classifier**: LightGBM คาดว่าจะชนะ MLP (ตามที่ CLAUDE.md §10 คาดไว้ — ไม่ใช่
  ความล้มเหลว) แต่ province embeddings ยังเป็น deliverable ที่ใช้ได้อิสระ
- **6-month forecast**: ดูผลจริงด้านบนว่า GRU ชนะ seasonal-naive หรือไม่ — รายงานตามจริง
  ทั้งสองทาง (CLAUDE.md Step 6)

**ข้อสรุปสำหรับ deploy** (Phase 7 dashboard): เลือกโมเดลที่ชนะ metric ของแต่ละงานจากตาราง
หัวข้อ 5 — ใช้ `rri.models.persistence` โหลดโมเดลที่บันทึกในหัวข้อ 7 ด้านล่าง

## 7. บันทึกโมเดลทั้งหมด (Phase 7 dashboard)

In [11]:
per.save_glm(glm_freq, freq_train, flood_fill, "glm_frequency")
per.save_glm(glm_sev, sev_train, flood_fill, "glm_severity")
per.save_lightgbm(lgbm_freq, "lightgbm_frequency")
per.save_lightgbm(lgbm_sev, "lightgbm_severity")
per.save_lightgbm(lgbm_clf, "lightgbm_classifier")
per.save_mlp(mlp_clf)
per.save_gru(gru_fitted)
print("saved 7 models to", settings.models_dir)
for p in sorted(settings.models_dir.iterdir()):
    print(" ", p.name, f"({p.stat().st_size / 1024:.0f} KB)")

saved 7 models to /Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/50_models
  glm_frequency.joblib (1723 KB)
  glm_severity.joblib (1896 KB)
  gru_forecast.pt (22 KB)
  lightgbm_classifier.joblib (2567 KB)
  lightgbm_frequency.joblib (768 KB)
  lightgbm_severity.joblib (107 KB)
  mlp_classifier.pt (201 KB)
